# 04 Ficher ellipses

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

In [ ]:
def plot_fisher_ellipses(
    param_names,
    theta_hat,
    cov,
    levels=(2.30, 6.17),  # 68%, 95%
    figsize=(6, 6),
):
    """
    Plot Fisher correlation ellipses from covariance matrix.

    Parameters
    ----------
    param_names : list of str
    theta_hat   : array-like (N,)
    cov         : array-like (N,N)
    levels      : tuple of Δχ² values
    """

    n = len(param_names)
    fig, axes = plt.subplots(n, n, figsize=(2 * n, 2 * n))

    # loop on parameters
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]

            # auto correlation
            if i == j:
                # 1D Gaussian
                sigma = np.sqrt(cov[i, i])
                x = np.linspace(theta_hat[i] - 4 * sigma, theta_hat[i] + 4 * sigma, 200)
                y = np.exp(-((x - theta_hat[i]) ** 2) / (2 * sigma**2))
                ax.plot(x, y)
                ax.set_yticks([])

            # ellipses : 2 parameter correlation
            elif i > j:
                # 2D ellipse
                # sub convariance 2x2
                sub_cov = cov[np.ix_([j, i], [j, i])]
                # diagonalisation
                vals, vecs = np.linalg.eigh(sub_cov)

                # tri des valeurs propres
                order = vals.argsort()[::-1]
                vals = vals[order]
                vecs = vecs[:, order]

                angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))

                for level in levels:
                    width = 2 * np.sqrt(vals[0] * level)
                    height = 2 * np.sqrt(vals[1] * level)

                    ellipse = Ellipse(
                        xy=(theta_hat[j], theta_hat[i]), width=width, height=height, angle=angle, fill=False
                    )
                    ax.add_patch(ellipse)

                ax.scatter(theta_hat[j], theta_hat[i], s=10)

            else:
                ax.axis("off")

            if i == n - 1:
                ax.set_xlabel(param_names[j])
            if j == 0:
                ax.set_ylabel(param_names[i])

    plt.tight_layout()
    return fig

In [ ]:
from scipy.stats import chi2

delta_chi2_68 = chi2.ppf(0.683, df=2)
delta_chi2_95 = chi2.ppf(0.954, df=2)

In [ ]:
print(delta_chi2_68, delta_chi2_95)

In [ ]:
param_names = ["Ωm", "σ8", "H0"]

theta_hat = np.array([0.3, 0.8, 70])

cov = np.array([[0.01, 0.005, 0.0], [0.005, 0.02, 0.0], [0.0, 0.0, 4.0]])

plot_fisher_ellipses(param_names, theta_hat, cov, levels=(delta_chi2_68, delta_chi2_95))
plt.show()